# Convert Data Units Using TEOS-10 Subroutines

## Import Data for Conversion


In [1]:
import pandas as pd
import gsw
import numpy as np

df = pd.read_parquet("/g/data/gv90/ad6790/amery-amisor/data/proc/01_imported.parquet")

## Handle Known Sensor Errors/Issues

### AM01: Pressure Sensor Failure

For AM01, pressure sensor data were bad for all 3 instruments, and a constant pressure value was selected for each instrument (from inspection of data near the start in January 2002), and this constant value was used to recalculate conductivity, and in the salinity calculation. Pressure column is -999.0 for all instruments. Assumed pressure of AM01 instruments is: \
cat1969 = 441.1 dbar \
cat1970 = 581.5 dbar \
cat1971 = 743.0 dbar

In [2]:
# Override each pressure for each respective cat with the constant values as above
# .loc is a property of pandas DataFrames that allows you to access and modify specific rows and columns based on labels or boolean conditions. 
# In this case, we are using .loc to select rows where the "borehole" column is "AM01" and the "cat" column matches the respective year (1969, 1970, 1971).
# We then assign the specified pressure values to those rows in the "pressure" column.
df.loc[(df["borehole"] == "AM01") & (df["cat"] == "1969"), "pressure"] = 441.1
df.loc[(df["borehole"] == "AM01") & (df["cat"] == "1970"), "pressure"] = 581.5
df.loc[(df["borehole"] == "AM01") & (df["cat"] == "1971"), "pressure"] = 743.0

### AM05: Pressure Sensor Failure

For AM05, pressure sensor data for the second and third downloads was bad for cat7327, and a constant pressure value was selected (the value immediately after deployment in December 2009), and this constant value was used to recalculate conductivity, and in the salinity calculation. Pressure column is -999.0 for all instruments. Assumed pressure of cat7327 at AM05 from 05/02/2011 12:00:01 onwards is: \
cat7327 = 580.2 dbar

In [3]:
# Override each pressure for the specific cat from the specified data onwards with the constant values as above
df.loc[(df["borehole"] == "AM05") & 
(df["cat"] == "7327") &
(df["time"] >= "2011-02-05 12:00:01"),
"pressure"] = 580.2

### AM06: Pressure Sensor Failure

For AM05, pressure sensor data for the second and third downloads was bad for cat7325, and a constant pressure value was selected (the value immediately after deployment), and this constant value was used to recalculate conductivity, and in the salinity calculation. Pressure column is -999.0 for all instruments. Assumed pressure of cat7325 at AM06 from 05/02/2011 09:30:01 onwards is: \
cat7325 = 551.3 dbar

In [4]:
# Override each pressure for the specific cat from the specified data onwards with the constant values as above
df.loc[(df["borehole"] == "AM06") & 
(df["cat"] == "7325") &
(df["time"] >= "2011-02-05 09:30:01"),
"pressure"] = 551.3

### Add Column Allocating Data to an Upper, Middle or Lower MicroCAT

Mainly for graphing purposes, it is important to retain information in the dataset to quickly allocate which MicroCATs belong to the upper, middle or lower mooring the water column. Here, we use pressure data, including the newly added assumed pressure due to bad sensors, to allocate as such.

In [5]:
df_sub = df.copy()

df["water_column_position"] = None

# Iterate over each borehole
for bh in df_sub["borehole"].unique():
    df_bh = df_sub[df_sub["borehole"] == bh]

    # Compute the median pressure for each cat at this particular borehole location
    # .groupby() groups data by "cat" such that each group contains all rows for a given mooring, selecting only the "pressure" column for analysis
    # .median() calculates the median for each cat
    # .sort_values() sorts the calculated medians in ascending order
    cat_pressure = (df_bh.groupby("cat")["pressure"].median().sort_values())
    # Extract the identifiers of each cat in the order sorted by median pressure
    cats_sorted = cat_pressure.index.tolist()

    # Assign labels based on position in the water column
    labels = ["upper", "middle", "lower"]

    # Since there are always exactly three cats, we can pair each cat with the corresponding label dependent on the sorted ordering
    mapping = dict(zip(cats_sorted, labels))
    
    # Apply this mapping to the dataframe
    mask = df["borehole"] == bh
    # df.loc[mask, "cat"].map(mapping) replaces each cat with its assigned label using the mapping dictionary
    # df.loc[mask, "water_column_position"] writes these lavels into the new column
    df.loc[mask, "water_column_position"] = df.loc[mask, "cat"].map(mapping)

## Conversions

### Practical Salinity to Absolute Salinity 

In [6]:
# Convert practical salinity to absolute salinity
df["absolute_salinity"] = gsw.conversions.SA_from_SP(
    df["practical_salinity"], # Practical salinity (PSS78)
    df["pressure"],           # Pressure (dbar)
    df["longitude"],          # Longitude of mooring
    df["latitude"]            # Latitude of mooring
)

df["absolute_salinity_derived"] = df["absolute_salinity"]

### AM02: Conductivity Recording Failure in the Bottom MicroCAT from March 2004

In [7]:
mask = ((df["borehole"] == "AM02") & (df["cat"] == "1174") & (df["time"] >= "2004-03-02 18:15:00"))

df.loc[mask, "absolute_salinity_derived"] = 34.7 

### AM02: Anomalous Salinity Recording in the Upper MicroCAT in September/October 2005

In [8]:
mask = ((df["borehole"] == "AM02") & (df["cat"] == "1623") & (df["time"] >= "2005-09-16 22:45:00") & (df["time"] <= "2005-10-05 22:15:00"))

df.loc[mask, "absolute_salinity"] = np.nan
df.loc[mask, "absolute_salinity_derived"] = 34.53333333 

### In Situ Temperature to Potential Temperature

In [9]:
df["potential_temperature"] = gsw.conversions.pt0_from_t(
    df["absolute_salinity_derived"],  # Absolute salinity (g/kg)
    df["in_situ_temperature"],# In-situ temperature (°C)
    df["pressure"]            # Pressure (dbar)
)

### In Situ Temperature to Conservative Temperature

In [10]:
df["conservative_temperature"] = gsw.conversions.CT_from_t(
    df["absolute_salinity_derived"],  # Absolute salinity (g/kg)
    df["in_situ_temperature"],# In-situ temperature (°C)
    df["pressure"]            # Pressure (dbar)
)

## Calculations

### In Situ Freezing Point Temperature

In [11]:
df["in_situ_freezing_point_temperature"] = gsw.t_freezing(
    df["absolute_salinity_derived"],  # Absolute salinity (g/kg)
    df["pressure"],           # Pressure (dbar)
    0                         # Saturation fraction (0 to 1, typically 0 for sub-ice shelf ocean cavitys, 1 for open ocean)
)

### Conservative Freezing Point Temperature

In [12]:
df["conservative_freezing_point_temperature"] = gsw.CT_freezing(
    df["absolute_salinity_derived"],  # Absolute salinity (g/kg)
    df["pressure"],           # Pressure (dbar)
    0                         # Saturation fraction (0 to 1, typically 0 for sub-ice shelf ocean cavitys, 1 for open ocean)
)

### Save Dataframe as Parquet

In [13]:
# Save to parquet file for efficient storage and future use
# Note: parquet is a columnar storage format that is more efficient for large datasets and preserves data types better than CSV
# Note: parquet files can be read back into pandas with pd.read_parquet("data/proc/02_converted.parquet")
# Note: this will overwrite any existing file at this location
df.to_parquet("/g/data/gv90/ad6790/amery-amisor/data/proc/02_converted.parquet", index=False)